# 009 — Entornos Python, Git y experimentos reproducibles

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Niveles de reproducibilidad:** 0 repetibilidad (yo, hoy) → 1 reproducibilidad (otro, con
mi código) → 2 replicabilidad (otro, con SU código) → 3 generalización. Los niveles 0-1
exigen fijar cinco cosas: código (Git), dependencias (lockfile), datos (versión+checksum),
aleatoriedad (semillas) y configuración.

- **venv:** intérprete y `site-packages` aislados por proyecto. `requirements.txt` declara
  rangos; el **lockfile** (`pip freeze`) congela versiones exactas — sin él, dos
  instalaciones en fechas distintas divergen.
- **Semillas:** los PRNG son deterministas; `seed` fija la secuencia. Cada biblioteca tiene
  su PRNG (sembrar `random` no siembra NumPy). La semilla da repetibilidad, **no validez**:
  se reporta sobre varias semillas.
- **Git:** commits = instantáneas inmutables identificadas por hash; el hash une el número
  del informe con el código exacto que lo produjo. Tags por experimento, `.gitignore` para
  datos y secretos.

Contrato experimental mínimo: `{commit, lockfile, semillas, parámetros, datos+checksum,
métrica, fecha}` — exactamente lo que el JSON del laboratorio implementa en miniatura.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Con la misma semilla los JSON son idénticos (repetibilidad, nivel 0);
con seed=99 cambian los valores muestreados pero no el contrato (claves, `kind`). La
semilla controla el PRNG; el código (biblioteca común) controla el contrato.

**Ejercicio 2.** (1) `git diff` entre estados → vacío, descarta código. (2) diff de
lockfiles → numpy difiere, candidato. (3) Recrear el venv con el lockfile antiguo y
re-ejecutar: si MAE vuelve a 12.3, causa confirmada — se cambió una sola variable y el
efecto siguió a esa variable.

**Ejercicio 3.** Coinciden exactamente: el PRNG es determinista y la semilla reinicia la
secuencia. `random` y `numpy.random` son generadores independientes: sembrar uno no afecta
al otro; hay que sembrar cada fuente usada.

**Ejercicio 4.** Ejemplo: `commit=a1b2c3d · env=requirements-lock.txt (sha256 9f3e…) ·
seed=1 · params=default · data=n/a · fecha=2026-07-29`. Desde el notebook no puedes
garantizar el commit real (el notebook puede estar sucio/no commiteado): por eso los
resultados citables se generan desde estados commiteados, no desde working trees.

In [ ]:
result = run_lab("observability", seed=9)
assert result["kind"] == "observability"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — repetibilidad verificada
r_a = run_lab("observability", seed=1)
r_b = run_lab("observability", seed=1)
r_c = run_lab("observability", seed=99)
print("misma semilla → idéntico:", r_a == r_b)
print("otra semilla → mismas claves:", sorted(r_a) == sorted(r_c))
assert r_a == r_b

In [ ]:
# Ejercicio 3 — el PRNG es determinista y por-biblioteca
import random

random.seed(42)
tanda1 = [random.random() for _ in range(3)]
random.seed(42)
tanda2 = [random.random() for _ in range(3)]
print("tandas idénticas:", tanda1 == tanda2)
assert tanda1 == tanda2
# numpy tiene su propio generador: random.seed NO lo siembra.
# La práctica correcta: rng = numpy.random.default_rng(42) por experimento.

## Reflexión

1. Ejecuta el laboratorio dos veces con la misma semilla y una con otra. ¿Qué nivel de
   reproducibilidad (0-3) acabas de demostrar y qué te faltaría para el siguiente nivel?
2. La métrica de un informe cambió sin que cambiara el código. Enumera, en orden de
   sospecha, los componentes del contrato que revisarías y el comando/diff con el que
   descartarías cada uno.
3. ¿Por qué "el notebook es el experimento" es una afirmación peligrosa, y cómo lo resuelve
   la arquitectura de este repositorio (`ai_evolution.labs` + notebooks finos)?